<!--
SPDX-FileCopyrightText: Copyright (c) 2026, NVIDIA CORPORATION & AFFILIATES.
All rights reserved.
SPDX-License-Identifier: Apache-2.0
-->

# Find the right page with ViDoRe HR

Build a searchable index of human resources documents, then see how often
NeMo Retriever finds the pages that answer each question.

**Choose your models → Ingest the pages → Search → See your scores**

You’ll get Recall and nDCG at **1, 3, 5, and 10**, plus a few example searches
you can inspect. Everything runs from the settings below.

> **Start here:** Use a Python 3.12 kernel in the repository root or `examples`.
> Choose local or hosted models, then select **Run All**. For a new environment,
> enable installation below; restart the kernel after the first installation.


## Choose how to run

| Mode | What you need | Where the models run |
| :--- | :--- | :--- |
| `"local"` | NVIDIA GPU(s) with enough memory for extraction and embedding | On your machine |
| `"hosted"` | An NVIDIA API key | Remote NVIDIA endpoints |

Start with **20 questions** to try the workflow. Set `MAX_QUERIES = None` when
you’re ready to score the full test split. Both runs search the complete corpus;
the first ingestion takes the most time, and later runs reuse the index.


In [ ]:
# These are the only settings you need for a typical run.
INFERENCE_MODE = "local"  # "local" or "hosted"
MAX_QUERIES = 20  # None = all questions in the test split.
OVERWRITE_INDEX = False  # True = rebuild the index.
INSTALL_DEPENDENCIES = False  # True for your first run in a new environment.

### Optional: models and storage

The next cell contains model names, endpoint URLs, storage paths, and retrieval
limits. Leave the defaults for a first run. Expand it to use your own compatible
NIM endpoints or pin a dataset revision. Indexes are kept separate by model and mode.


In [ ]:
from pathlib import Path

DATASET_REVISION = "main"  # Pin a Hugging Face commit SHA for repeatable runs.
LOCAL_EMBED_MODEL = "nvidia/Nemotron-3-Embed-1B-BF16"
HOSTED_EMBED_MODEL = "nvidia/nemotron-3-embed-1b"
PAGE_ELEMENTS_URL = "https://ai.api.nvidia.com/v1/cv/nvidia/nemotron-page-elements-v3"
OCR_URL = "https://ai.api.nvidia.com/v1/cv/nvidia/nemotron-ocr-v2"
EMBED_URL = "https://integrate.api.nvidia.com/v1/embeddings"

QUERY_BATCH_SIZE = 32
INITIAL_CANDIDATES = 40
MAX_CANDIDATES = 1000
CUTOFFS = (1, 3, 5, 10)

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "nemo_retriever" / "pyproject.toml").is_file():
    REPO_ROOT = REPO_ROOT.parent
assert (
    REPO_ROOT / "nemo_retriever" / "pyproject.toml"
).is_file(), "Start in the repository root or examples."
DATA_DIR = REPO_ROOT / "data" / "vidore_v3_hr_notebook"
INDEX_DIR = REPO_ROOT / "examples" / "lancedb-vidore-v3-hr"
RESULTS_DIR = REPO_ROOT / "examples" / "vidore-v3-hr-results"

assert INFERENCE_MODE in {"local", "hosted"}, 'Choose INFERENCE_MODE="local" or "hosted".'
assert MAX_QUERIES is None or (isinstance(MAX_QUERIES, int) and MAX_QUERIES > 0)
assert QUERY_BATCH_SIZE > 0 and 10 <= INITIAL_CANDIDATES <= MAX_CANDIDATES
EMBED_MODEL = LOCAL_EMBED_MODEL if INFERENCE_MODE == "local" else HOSTED_EMBED_MODEL

## Get ready

This cell checks your environment. If you selected hosted models and have not set
`NVIDIA_API_KEY`, you’ll get a hidden key prompt. Hosted mode sends page content and
questions to your endpoints. Local mode checks that a CUDA GPU is available.

The following collapsed cell defines the index, search, and report helpers used below.
You can expand it to inspect the implementation; the walkthrough cells show each stage.

In [ ]:
import os
import shutil
import subprocess
import sys

if INSTALL_DEPENDENCIES:
    target = str(REPO_ROOT / "nemo_retriever")
    if INFERENCE_MODE == "local":
        target += "[local]"
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-e",
            target,
            "datasets>=4.8.2",
            "huggingface-hub>=1.5,<2",
        ],
        check=True,
    )
    print(
        "Dependencies installed. Restart the kernel before your next run, and turn installation off."
    )

import hashlib
import json
import lancedb
import pandas as pd
from datasets import load_dataset
from huggingface_hub import HfApi
from IPython.display import HTML, display
from tqdm.auto import tqdm
from nemo_retriever.graph.retriever import Retriever
from nemo_retriever.tools.recall.beir import (
    build_queries_by_id,
    build_qrels_by_query_id,
    build_beir_run_from_hits,
    compute_beir_metrics,
)

os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ.get("PATH", "")
retriever_cli = shutil.which("retriever")
assert retriever_cli, "Install nemo-retriever in this kernel environment first."
if INFERENCE_MODE == "local":
    import torch

    assert (
        torch.cuda.is_available()
    ), "No NVIDIA GPU found. Choose hosted mode, or use a CUDA-enabled kernel."
    print(f"Ready · Local models · {torch.cuda.device_count()} GPU(s)")
else:
    from getpass import getpass

    if not os.environ.get("NVIDIA_API_KEY") and "api.nvidia.com" in EMBED_URL:
        os.environ["NVIDIA_API_KEY"] = getpass("NVIDIA API key (hidden): ")
    print("Ready · Hosted models")
print(f"Embedding model: {EMBED_MODEL}")

In [ ]:
def prepare_index():
    index_config = {
        "dataset": DATASET_ID,
        "revision": dataset_sha,
        "mode": INFERENCE_MODE,
        "embedding_model": EMBED_MODEL,
        "page_elements_url": PAGE_ELEMENTS_URL if INFERENCE_MODE == "hosted" else None,
        "ocr_url": OCR_URL if INFERENCE_MODE == "hosted" else None,
        "embed_url": EMBED_URL if INFERENCE_MODE == "hosted" else None,
        "corpus_format": "single-page-image-pdf-150dpi-v1",
    }
    config_hash = hashlib.sha256(json.dumps(index_config, sort_keys=True).encode()).hexdigest()[:12]
    lancedb_uri = INDEX_DIR / f"{INFERENCE_MODE}-{config_hash}"
    lancedb_uri.mkdir(parents=True, exist_ok=True)
    table_name = "vidore_v3_hr"
    manifest_path = lancedb_uri / "notebook-index.json"
    database = lancedb.connect(str(lancedb_uri))
    table_exists = table_name in database.list_tables(limit=None).tables
    if table_exists and not OVERWRITE_INDEX:
        assert (
            manifest_path.is_file()
        ), "Index has no completed-ingestion manifest. Set OVERWRITE_INDEX=True."
        assert (
            json.loads(manifest_path.read_text()) == index_config
        ), "Index configuration differs. Set OVERWRITE_INDEX=True."
        print("Using your existing index.")
    else:
        # Remove the completion marker before rebuilding so failed runs cannot be reused.
        manifest_path.unlink(missing_ok=True)
        command = [
            retriever_cli,
            "ingest",
            "batch",
            str(corpus_dir),
            "--lancedb-uri",
            str(lancedb_uri),
            "--table-name",
            table_name,
            "--embed-model-name",
            EMBED_MODEL,
        ]
        if INFERENCE_MODE == "local":
            command += ["--local-ingest-embed-backend", "hf"]
        else:
            command += [
                "--page-elements-invoke-url",
                PAGE_ELEMENTS_URL,
                "--ocr-invoke-url",
                OCR_URL,
                "--embed-invoke-url",
                EMBED_URL,
            ]
        if table_exists:
            command.append("--overwrite")
        print(f"Ingesting {len(corpus_ids):,} pages with {INFERENCE_MODE} models…")
        subprocess.run(command, check=True)
        database = lancedb.connect(str(lancedb_uri))
        assert (
            database.open_table(table_name).count_rows() > 0
        ), "Ingestion produced an empty table."
        manifest_path.write_text(json.dumps(index_config, indent=2) + "\n")
    row_count = database.open_table(table_name).count_rows()
    assert row_count > 0, "The retrieval table is empty."
    print(f"Index ready · {row_count:,} searchable elements")
    return database, lancedb_uri, table_name, index_config, config_hash, row_count


def page_run(ids, hits):
    result = build_beir_run_from_hits(ids, hits, doc_id_field="pdf_basename")
    for qid, query_hits in zip(ids, hits):
        for hit in query_hits:
            single = build_beir_run_from_hits([qid], [[hit]], doc_id_field="pdf_basename")[qid]
            assert (
                single and set(single) <= corpus_ids
            ), f"Unrecognized corpus page metadata for query {qid}."
    return result


def retrieve_pages():
    run = {}
    retrieval_depths = {}
    candidate_limit = min(MAX_CANDIDATES, row_count)
    target_pages = min(max(CUTOFFS), len(corpus_ids))
    for start in tqdm(range(0, len(queries), QUERY_BATCH_SIZE), desc="Searching"):
        ids = query_ids[start : start + QUERY_BATCH_SIZE]
        texts = dict(zip(ids, queries[start : start + QUERY_BATCH_SIZE]))
        pending = ids
        depth = min(INITIAL_CANDIDATES, candidate_limit)
        while pending:
            hits = retriever.queries([texts[qid] for qid in pending], top_k=depth)
            assert len(hits) == len(pending), "Retriever returned an unexpected number of results."
            batch_run = page_run(pending, hits)
            run.update(batch_run)
            retrieval_depths.update({qid: depth for qid in pending})
            pending = [qid for qid in pending if len(run[qid]) < target_pages]
            if depth >= candidate_limit:
                break
            depth = min(depth * 2, candidate_limit)
    insufficient = sum(len(run[qid]) < target_pages for qid in query_ids)
    if insufficient:
        print(
            f"{insufficient} queries returned fewer than {target_pages} unique pages at the candidate limit "
            f"({candidate_limit}); empty or short rankings are still scored. Increase MAX_CANDIDATES if needed."
        )
    return run, retrieval_depths, insufficient


def score_queries():
    metrics = compute_beir_metrics(qrels, run, ks=CUTOFFS)
    summary_df = pd.DataFrame(
        [{"k": k, "Recall": metrics[f"recall@{k}"], "nDCG": metrics[f"ndcg@{k}"]} for k in CUTOFFS]
    ).set_index("k")
    rows = []
    for qid, query in zip(query_ids, queries):
        rows.append(
            {
                "query_id": qid,
                "query": query,
                "relevant_pages": [doc_id for doc_id, score in qrels[qid].items() if score > 0],
                "ranked_pages": list(run[qid])[: max(CUTOFFS)],
                "unique_pages_retrieved": len(run[qid]),
                "candidates_requested": retrieval_depths[qid],
                **compute_beir_metrics({qid: qrels[qid]}, {qid: run[qid]}, ks=CUTOFFS),
            }
        )
    per_query_df = pd.DataFrame(rows)
    return summary_df, per_query_df


def show_examples(results, pages, count=3):
    """Show a few readable searches without overwhelming the results table."""
    from html import escape

    page_labels = {
        str(row["corpus_id"]): (
            f"{row.get('doc_id', 'Corpus page').replace('_', ' ')}"
            f" · page {int(row['page_number_in_doc']) + 1}"
            if row.get("page_number_in_doc") is not None
            else f"Corpus page {row['corpus_id']}"
        )
        for row in pages.select_columns(
            [
                name
                for name in ("corpus_id", "doc_id", "page_number_in_doc")
                if name in pages.column_names
            ]
        )
    }
    examples = []
    for _, row in results.head(count).iterrows():
        relevant = set(row["relevant_pages"])
        ranking = (
            "".join(
                f"<li>{escape(page_labels[page_id])} "
                f"<strong>{'✓ Relevant' if page_id in relevant else '— Unlabeled'}</strong></li>"
                for page_id in row["ranked_pages"]
            )
            or "<li>No pages retrieved.</li>"
        )
        examples.append(
            '<details style="margin:12px 0;padding:12px;border:1px solid #aaa;border-radius:8px">'
            f'<summary style="cursor:pointer">{escape(row["query"])}</summary>'
            f'<p>Recall@10: {row["recall@10"]:.0%} · nDCG@10: {row["ndcg@10"]:.0%}</p>'
            f"<ol>{ranking}</ol></details>"
        )
    display(HTML("".join(examples)))


def save_reports():
    from datetime import datetime, timezone

    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    output_dir = RESULTS_DIR / f"{INFERENCE_MODE}-{config_hash}-{timestamp}"
    output_dir.mkdir(parents=True, exist_ok=False)
    summary_df.to_csv(output_dir / "summary.csv")
    per_query_df.to_csv(output_dir / "per_query.csv", index=False)
    (output_dir / "run.json").write_text(json.dumps(run, indent=2) + "\n")
    (output_dir / "config.json").write_text(
        json.dumps(
            {
                **index_config,
                "max_queries": MAX_QUERIES,
                "evaluated_queries": len(query_ids),
                "cutoffs": CUTOFFS,
                "query_batch_size": QUERY_BATCH_SIZE,
                "initial_candidates": INITIAL_CANDIDATES,
                "max_candidates": MAX_CANDIDATES,
                "queries_below_target_pages": insufficient,
            },
            indent=2,
        )
        + "\n"
    )
    return output_dir

## 1. Meet the dataset

[ViDoRe v3 HR](https://huggingface.co/datasets/vidore/vidore_v3_hr) pairs document
pages with questions and relevance labels: the pages a good search should find.
We download the test split and show a sample before starting ingestion.

Hugging Face caches the download. If access requires authentication, set `HF_TOKEN`
or log in to Hugging Face before running. All partitions use the same dataset revision.


In [ ]:
DATASET_ID = "vidore/vidore_v3_hr"
dataset_sha = HfApi().dataset_info(DATASET_ID, revision=DATASET_REVISION).sha
partitions = {
    name: load_dataset(DATASET_ID, data_dir=name, split="test", revision=dataset_sha)
    for name in ("corpus", "queries", "qrels")
}
corpus = partitions["corpus"]
corpus_ids = {str(value) for value in corpus["corpus_id"]}
assert len(corpus_ids) == len(corpus), "Corpus IDs must be unique."
assert all(value.isdecimal() for value in corpus_ids), "Expected numeric ViDoRe corpus IDs."
query_ids, queries = build_queries_by_id(partitions["queries"])
qrels = build_qrels_by_query_id(partitions["qrels"], allowed_query_ids=set(query_ids))
unknown_ids = {doc_id for gold in qrels.values() for doc_id in gold} - corpus_ids
assert not unknown_ids, f"Qrels reference unknown corpus IDs: {sorted(unknown_ids)[:5]}"
selected = [
    (qid, query)
    for qid, query in zip(query_ids, queries)
    if any(score > 0 for score in qrels.get(qid, {}).values())
]
if MAX_QUERIES is not None:
    selected = selected[:MAX_QUERIES]
assert selected, "No queries with positive relevance labels found."
query_ids, queries = map(list, zip(*selected))
assert len(set(query_ids)) == len(query_ids), "Query IDs must be unique."
qrels = {qid: qrels[qid] for qid in query_ids}
print(f"{len(corpus):,} pages ready · {len(queries):,} questions selected")
print(f"Try this question: {queries[0]}")
display(
    corpus[0]["image"]
    .copy()
    .resize((360, round(360 * corpus[0]["image"].height / corpus[0]["image"].width)))
)

### Turn pages into ingestion inputs

Each page image becomes a one-page PDF. Its filename preserves the corpus page ID,
so we can match search results to the relevance labels exactly. Previously exported
pages are reused. We ingest the images rather than the dataset’s reference markdown.


In [ ]:
corpus_dir = DATA_DIR / dataset_sha / "pdfs"
corpus_dir.mkdir(parents=True, exist_ok=True)
missing_pages = [
    i for i, value in enumerate(corpus["corpus_id"]) if not (corpus_dir / f"{value}.pdf").is_file()
]
for row in tqdm(
    corpus.select(missing_pages), total=len(missing_pages), desc="Exporting corpus pages"
):
    image = row["image"].convert("RGB")
    path = corpus_dir / f"{row['corpus_id']}.pdf"
    temporary = path.with_suffix(".tmp")
    image.save(temporary, format="PDF", resolution=150.0)
    temporary.replace(path)
assert {path.stem for path in corpus_dir.glob("*.pdf")} == corpus_ids
print(f"Corpus prepared · {len(corpus_ids):,} pages")

## 2. Build your search index

NeMo Retriever extracts the page content, embeds it with your selected model,
and stores it in LanceDB. **This is the slowest step on the first run.**
The ingestion log below shows progress; once the index exists, this step reuses it.

Set `OVERWRITE_INDEX=True` to start fresh. Model or endpoint changes get their own
index automatically. An interrupted ingestion must be rebuilt before reuse.


In [ ]:
database, lancedb_uri, table_name, index_config, config_hash, row_count = prepare_index()

## 3. Search for the answers

We embed each question with the same model used for ingestion, then search the index.
A page may produce several extracted elements; we keep its first appearance so the
ranking contains **unique pages**. If needed, we search more elements to fill the top ten.

The progress bar tracks batches of questions. Empty or short rankings still count
in the scores, and unrecognized page IDs stop the run so they can be checked.


In [ ]:
embed_kwargs = {"model_name": EMBED_MODEL, "embed_model_name": EMBED_MODEL}
if INFERENCE_MODE == "local":
    embed_kwargs["local_ingest_embed_backend"] = "hf"
else:
    embed_kwargs.update(embedding_endpoint=EMBED_URL, api_key=os.environ.get("NVIDIA_API_KEY", ""))
retriever = Retriever(
    run_mode="local" if INFERENCE_MODE == "local" else "service",
    vdb_kwargs={"uri": str(lancedb_uri), "table_name": table_name},
    embed_kwargs=embed_kwargs,
    top_k=INITIAL_CANDIDATES,
)

run, retrieval_depths, insufficient = retrieve_pages()
print(f"Search complete · {len(query_ids):,} questions")

## 4. How did retrieval do?

| Metric | What it tells you |
| :--- | :--- |
| **Recall@k** | What fraction of the relevant pages appeared in the first k results? |
| **nDCG@k** | How well did the search put the most relevant pages first? |

Higher is better for both. Scores are averaged across all selected questions.
nDCG uses the relevance grades in the labels and discounts results farther down the ranking.
A 20-question run is a preview; use the full test split for your benchmark result.

Expand an example below to see which retrieved pages were labeled relevant.


In [ ]:
summary_df, per_query_df = score_queries()
print(f"{len(query_ids):,} questions · {INFERENCE_MODE.title()} models · ViDoRe HR")
display(
    summary_df.style.format("{:.1%}")
    .bar(subset=["Recall", "nDCG"], color="#d8eee4", vmin=0, vmax=1)
    .set_caption("Retrieval quality · higher is better")
)
show_examples(per_query_df, corpus)

### Keep your results

Save the scores, per-question rankings, and run settings for comparison.
You can also explore `per_query_df` directly in the notebook.


In [ ]:
output_dir = save_reports()
print(f"Reports saved to {output_dir}")